# How does the number of classes affect softmax and SDM? CIFAR-10 against Imagenette

On CIFAR-100, SDM formed no region on the 100 classes; on Imagenette's 10 classes it did ([notebook 10](10_summary_imagenette_vs_cifar100.ipynb)). This notebook tests the number of classes directly, on two 10-class datasets at the same resolution:

- **CIFAR-10** (32 × 32) and **Imagenette at 32 × 32**, with the **same pipeline**: images stored at 36 × 36, random 32 × 32 crops for training, the central 32 × 32 for evaluation.
- **The same amount of data per class:** 500 training, 200 calibration and 150 validation images per class in both datasets, with a split fixed once per dataset, so an image keeps its role at every K.
- **Nested class sets, K = 2 … 10:** each step keeps every class of the previous step and adds **one randomly chosen new class**. A *chain* is one such random order; there are 3 chains, each with its own training seed, so a jump at some K can be traced to the class added there, and the mean over chains shows the effect of the number of classes itself.
- **At every step:** a fresh ResNet-18 (notebook 08's recipe), a temperature and SDM fitted on calibration, all numbers measured on validation.
- **A free out-of-distribution test:** for K < 10, the validation images of the classes *not yet added* were never seen in training. Any answer on them is wrong, so we can measure how often softmax and SDM wrongly answer them.

The runs are produced by `experiments/class_sweep.py` (about 3 hours for all 54 runs on an Apple M3 Pro, resumable):

```bash
python experiments/class_sweep.py --dataset cifar10 imagenette --chains 3 --k 2-10
```

This notebook only reads the results in `outputs/class_sweep/`. Validation has 150 images per class (300 at K = 2), so per-run numbers carry noticeable noise; the bands show the spread over chains.

## 1. Load the results

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

REPO_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src/sdm_vision").is_dir())
sys.path.insert(0, str(REPO_ROOT / "src"))
from sdm_vision.metrics import EVALUATION_VERSION

RESULTS_DIR = Path("../outputs/class_sweep")
DATASETS = {"cifar10": "CIFAR-10", "imagenette": "Imagenette 32 × 32"}

# One fixed color per dataset in every chart (categorical slots 1 and 2 of the reference palette; orange is
# Imagenette 32 x 32 as in notebook 10). Line style carries the method: solid = softmax, dashed = SDM.
COLORS = {"CIFAR-10": "#2a78d6", "Imagenette 32 × 32": "#eb6834"}
INK, INK_SECONDARY, INK_MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": AXIS, "axes.labelcolor": INK_SECONDARY,
    "axes.titlecolor": INK, "axes.titlesize": 11, "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "legend.frameon": False, "legend.fontsize": 8, "lines.linewidth": 2,
})

runs = pd.DataFrame([json.loads(path.read_text()) for path in sorted(RESULTS_DIR.glob("*/chain*_K*/metrics.json"))])
assert len(runs), f"no results in {RESULTS_DIR}: run experiments/class_sweep.py first"
if "evaluation_version" not in runs or not runs["evaluation_version"].eq(EVALUATION_VERSION).all():
    raise ValueError("Legacy class-sweep metrics: preserve the historical runs and generate corrected runs in a new output directory, then set RESULTS_DIR to it.")
runs["dataset"] = runs["dataset"].map(DATASETS)
runs = runs.sort_values(["dataset", "chain", "k"]).reset_index(drop=True)

expected = {(DATASETS[d], c, k) for d in DATASETS for c in range(runs["chain"].max() + 1) for k in range(2, 11)}
missing = sorted(expected - set(zip(runs["dataset"], runs["chain"], runs["k"])))
print(f"{len(runs)} runs loaded" + (f"; missing {len(missing)}: {missing[:6]}{' ...' if len(missing) > 6 else ''}" if missing else "; sweep complete"))

### The class each chain adds at each step

Read across a row: the classes of the K-class task are the first K columns. The first two columns are the starting pair.

In [ ]:
orders = (runs[runs["k"] == runs.groupby(["dataset", "chain"])["k"].transform("max")]
          .set_index(["dataset", "chain"]).apply(lambda row: pd.Series(row["classes"] + row["excluded"]), axis=1))
orders.columns = [f"#{i + 1}" for i in orders.columns]
orders

### Plotting helpers

`plot_vs_k` draws, for each dataset, the **mean over chains** as a line and the **min–max over chains** as a shaded band.

In [ ]:
def by_k(column):
    return runs.groupby(["dataset", "k"])[column].agg(["mean", "min", "max"]).reset_index()


def plot_vs_k(ax, column, linestyle="-", band=True):
    for dataset, group in by_k(column).groupby("dataset"):
        ax.plot(group["k"], group["mean"], color=COLORS[dataset], linestyle=linestyle, marker="o", markersize=5,
                markeredgecolor=SURFACE, markeredgewidth=1.5)
        if band:
            ax.fill_between(group["k"], group["min"], group["max"], color=COLORS[dataset], alpha=0.12, linewidth=0)


def legend(ax, styles=(("-", "softmax"), ("--", "SDM")), loc="best"):
    handles = [Line2D([], [], color=color, linewidth=2, label=dataset) for dataset, color in COLORS.items()]
    handles += [Line2D([], [], color=INK_MUTED, linewidth=2, linestyle=style, label=label) for style, label in styles]
    ax.legend(handles=handles, loc=loc)


def percent(ax, axis="y"):
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))


def k_axis(ax):
    ax.set_xticks(range(2, 11))
    ax.set_xlabel("number of classes K")


def table(columns, formats=None):
    # Mean over chains, one row per K and one column block per dataset.
    formats = formats or {}
    out = runs.groupby(["k", "dataset"])[list(columns)].mean().unstack("dataset")
    styles = {f"{columns[metric]} · {dataset}": formats.get(metric, "{:.1%}") for metric, dataset in out.columns}  # keyed by metric
    out.columns = list(styles)
    return out.style.format(styles, na_rep="-")

## 2. Accuracy

Raw accuracy falls with K for any classifier simply because there are more ways to be wrong (chance is 1/K, the dotted gray line). **Chance-adjusted accuracy**, (accuracy − 1/K) / (1 − 1/K), is 0 at chance and 1 when perfect, so it is comparable across K.

In [ ]:
fig, (ax_raw, ax_adj) = plt.subplots(1, 2, figsize=(12, 4.2))
plot_vs_k(ax_raw, "val_accuracy")
ks = np.arange(2, 11)
ax_raw.plot(ks, 1 / ks, color=INK_MUTED, linestyle=":", linewidth=1.5)
ax_raw.annotate("chance (1/K)", (ks[-1], 1 / ks[-1]), xytext=(-4, 6), textcoords="offset points", ha="right", fontsize=8, color=INK_SECONDARY)
ax_raw.set(title="Validation accuracy", ylabel="accuracy", ylim=(0, 1.02))
plot_vs_k(ax_adj, "chance_adjusted_accuracy")
ax_adj.set(title="Chance-adjusted accuracy", ylabel="(accuracy − 1/K) / (1 − 1/K)", ylim=(0, 1.02))
for ax in (ax_raw, ax_adj):
    k_axis(ax)
    percent(ax)
    legend(ax, styles=(), loc="lower left")
plt.tight_layout()
plt.show()

table({"val_accuracy": "validation", "chance_adjusted_accuracy": "chance-adjusted", "train_accuracy": "train"})

## 3. Calibration

Expected calibration error (15 bins) on validation, of the raw softmax (dotted) and after temperature scaling (solid), and the fitted temperature (above 1 means the raw softmax was overconfident).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
plot_vs_k(ax, "ece_raw", linestyle=":")
plot_vs_k(ax, "ece_scaled", linestyle="-")
ax.set(title="Calibration error on validation", ylabel="ECE", ylim=(0, None))
k_axis(ax)
percent(ax)
legend(ax, styles=((":", "raw softmax (T = 1)"), ("-", "temperature-scaled")), loc="upper left")
plt.tight_layout()
plt.show()

table({"ece_raw": "ECE raw", "ece_scaled": "ECE scaled", "temperature": "temperature"}, {"temperature": "{:.2f}"})

## 4. Coverage at 95% accuracy: softmax against SDM

- **Softmax:** the temperature-scaled confidence threshold chosen on calibration for 95% selective accuracy (plain rule, as in notebook 04).
- **SDM:** the images inside a region at α ≥ 0.95 (0 if SDM formed no such region).

Left: the share of validation images each rule answers. Right: how accurate those answers are; the dotted line is the 95% target. A rule that answers nothing has no accuracy, so its point is missing.

In [ ]:
fig, (ax_cov, ax_acc) = plt.subplots(1, 2, figsize=(12, 4.2))
plot_vs_k(ax_cov, "softmax_plain_95_coverage")
plot_vs_k(ax_cov, "sdm_coverage_095", linestyle="--")
ax_cov.set(title="Share of validation images answered", ylabel="coverage", ylim=(0, 1.02))
plot_vs_k(ax_acc, "softmax_plain_95_accuracy", band=False)
plot_vs_k(ax_acc, "sdm_accuracy_095", linestyle="--", band=False)
ax_acc.axhline(0.95, color=INK_MUTED, linestyle=":", linewidth=1.5, zorder=0)
ax_acc.set(title="Accuracy of the answered images", ylabel="selective accuracy", ylim=(0.85, 1.005))
for ax in (ax_cov, ax_acc):
    k_axis(ax)
    percent(ax)
legend(ax_cov, styles=(("-", "softmax 95% threshold"), ("--", "SDM α ≥ 0.95")), loc="upper right")
plt.tight_layout()
plt.show()

table({"softmax_plain_95_coverage": "softmax coverage", "softmax_plain_95_accuracy": "softmax accuracy",
       "sdm_coverage_095": "SDM α≥0.95 coverage", "sdm_accuracy_095": "SDM α≥0.95 accuracy"})

## 5. Where SDM forms regions

SDM forms a region at α only if, at one similarity cutoff, **every** class keeps calibration images and nearly all of them get at least α probability for their true class. Left: the highest α at which SDM formed a region, per chain (a hollow marker at the bottom means no region at all). Right: the share of validation images inside any region.

In [ ]:
fig, (ax_alpha, ax_any) = plt.subplots(1, 2, figsize=(12, 4.2))
offsets = {dataset: offset for dataset, offset in zip(COLORS, [-0.12, 0.12])}
for (dataset, chain), group in runs.groupby(["dataset", "chain"]):
    x = group["k"] + offsets[dataset] + (chain - 1) * 0.04
    formed = group["sdm_top_alpha"].notna()
    ax_alpha.scatter(x[formed], group["sdm_top_alpha"][formed], s=40, color=COLORS[dataset], edgecolor=SURFACE, linewidth=1.5, zorder=3)
    ax_alpha.scatter(x[~formed], np.full((~formed).sum(), 0.5), s=40, facecolor="none", edgecolor=COLORS[dataset], linewidth=1.5, zorder=3)
ax_alpha.set(title="Highest α with a region (one marker per chain)", ylabel="α", ylim=(0.47, 0.97))
ax_alpha.set_yticks([0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95], labels=["none"] + [f"{a:.2f}" for a in np.arange(0.55, 0.96, 0.05)])
plot_vs_k(ax_any, "sdm_coverage_any", linestyle="--")
ax_any.set(title="Share of validation images in any SDM region", ylabel="coverage", ylim=(0, 1.02))
percent(ax_any)
for ax in (ax_alpha, ax_any):
    k_axis(ax)
legend(ax_alpha, styles=(), loc="lower right")
legend(ax_any, styles=(("--", "SDM"),), loc="lower left")
plt.tight_layout()
plt.show()

(runs.assign(regions=runs["sdm_regions"].map(lambda r: ", ".join(f"{a:.2f}" for a in r) or "none"))
     .pivot_table(index="k", columns=["dataset", "chain"], values="regions", aggfunc="first"))

## 6. Ranking: does SDM order right and wrong answers better?

**E-AURC** is the area under the risk–coverage curve minus that of a perfect ranking with the same mistakes: 0 means right answers always outrank wrong ones. Unlike plain AURC, it does not shrink just because accuracy is higher, so it is comparable across K. Lower is better.

Right: the paired difference **softmax AURC − SDM AURC** for each run, with its 95% bootstrap interval (1,000 resamples of the validation images). Above 0 means SDM ranks better; an interval that crosses 0 means no clear difference.

In [ ]:
fig, (ax_eaurc, ax_diff) = plt.subplots(1, 2, figsize=(12, 4.2))
plot_vs_k(ax_eaurc, "softmax_e_aurc")
plot_vs_k(ax_eaurc, "sdm_e_aurc", linestyle="--")
ax_eaurc.set(title="Excess AURC (lower is better)", ylabel="E-AURC", ylim=(0, None))
legend(ax_eaurc, loc="upper left")

for (dataset, chain), group in runs.groupby(["dataset", "chain"]):
    x = group["k"] + offsets[dataset] + (chain - 1) * 0.04
    low = group["aurc_softmax_minus_sdm_interval"].map(lambda i: i[0])
    high = group["aurc_softmax_minus_sdm_interval"].map(lambda i: i[1])
    value = group["aurc_softmax_minus_sdm"]
    ax_diff.errorbar(x, value, yerr=[value - low, high - value], fmt="o", markersize=4, color=COLORS[dataset],
                     ecolor=COLORS[dataset], elinewidth=1, capsize=0, alpha=0.9)
ax_diff.axhline(0, color=INK_MUTED, linewidth=1)
ax_diff.text(0.01, 0.97, "above 0: SDM ranks better", transform=ax_diff.transAxes, fontsize=8, color=INK_SECONDARY, va="top")
ax_diff.text(0.01, 0.03, "below 0: softmax ranks better", transform=ax_diff.transAxes, fontsize=8, color=INK_SECONDARY)
ax_diff.set(title="Softmax AURC − SDM AURC, per run (95% bootstrap interval)", ylabel="AURC difference")
legend(ax_diff, styles=(), loc="lower right")
for ax in (ax_eaurc, ax_diff):
    k_axis(ax)
plt.tight_layout()
plt.show()

table({"softmax_e_aurc": "softmax E-AURC", "sdm_e_aurc": "SDM E-AURC", "softmax_auroc": "softmax AUROC", "sdm_auroc": "SDM AUROC"},
      {"softmax_e_aurc": "{:.3f}", "sdm_e_aurc": "{:.3f}", "softmax_auroc": "{:.3f}", "sdm_auroc": "{:.3f}"})

## 7. Calibration-matched coverage comparison

For every run where SDM formed a region, compare its top-region accuracy with softmax using a threshold matched to SDM coverage **on calibration only**. Equal-confidence groups enter together, so the attainable softmax calibration coverage may be lower. Validation coverages can differ; this is not an exact validation-coverage comparison. The intervals are paired bootstrap intervals over validation images, conditional on the fitted rules.


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
has_region = runs["matched_accuracy_sdm_minus_softmax"].notna() if "matched_accuracy_sdm_minus_softmax" in runs else pd.Series(False, index=runs.index)
for (dataset, chain), group in runs[has_region].groupby(["dataset", "chain"]):
    x = group["k"] + offsets[dataset] + (chain - 1) * 0.04
    value = group["matched_accuracy_sdm_minus_softmax"]
    low = group["matched_accuracy_sdm_minus_softmax_interval"].map(lambda i: i[0])
    high = group["matched_accuracy_sdm_minus_softmax_interval"].map(lambda i: i[1])
    ax.errorbar(x, value, yerr=[value - low, high - value], fmt="o", markersize=4, color=COLORS[dataset],
                ecolor=COLORS[dataset], elinewidth=1, capsize=0, alpha=0.9)
ax.axhline(0, color=INK_MUTED, linewidth=1)
ax.text(0.01, 0.97, "above 0: SDM more accurate", transform=ax.transAxes, fontsize=8, color=INK_SECONDARY, va="top")
ax.text(0.01, 0.03, "below 0: softmax more accurate", transform=ax.transAxes, fontsize=8, color=INK_SECONDARY)
ax.set(title="SDM top region − calibration-matched softmax", ylabel="accuracy difference")
k_axis(ax)
percent(ax)
legend(ax, styles=(), loc="lower right")
plt.tight_layout()
plt.show()

table({"sdm_top_alpha": "SDM top α", "sdm_coverage_top": "coverage", "sdm_accuracy_top": "SDM accuracy",
       "matched_softmax_accuracy": "softmax accuracy"}, {"sdm_top_alpha": "{:.2f}"}) if has_region.any() else print("SDM formed no region in any run.")

## 8. Images from classes the model never saw

For K < 10, the validation images of the 10 − K classes not yet added are **out-of-distribution**: the model has to put them in one of its K classes, so every answer on them is wrong.

Left: the share of those images each rule answers anyway (lower is better): the softmax 95% threshold, and SDM at α ≥ 0.95. Right: how well each score separates in-set images (high) from excluded-class images (low), as an AUROC (0.5 = no separation, 1 = perfect).

In [ ]:
ood = runs[runs["k"] < 10]
fig, (ax_answered, ax_auroc) = plt.subplots(1, 2, figsize=(12, 4.2))
for column, style in [("ood_softmax_answered_plain_95", "-"), ("ood_sdm_answered_095", "--")]:
    for dataset, group in ood.groupby(["dataset", "k"])[column].agg(["mean", "min", "max"]).reset_index().groupby("dataset"):
        ax_answered.plot(group["k"], group["mean"], color=COLORS[dataset], linestyle=style, marker="o", markersize=5, markeredgecolor=SURFACE, markeredgewidth=1.5)
        ax_answered.fill_between(group["k"], group["min"], group["max"], color=COLORS[dataset], alpha=0.12, linewidth=0)
for column, style in [("ood_softmax_auroc", "-"), ("ood_sdm_auroc", "--")]:
    for dataset, group in ood.groupby(["dataset", "k"])[column].agg(["mean", "min", "max"]).reset_index().groupby("dataset"):
        ax_auroc.plot(group["k"], group["mean"], color=COLORS[dataset], linestyle=style, marker="o", markersize=5, markeredgecolor=SURFACE, markeredgewidth=1.5)
        ax_auroc.fill_between(group["k"], group["min"], group["max"], color=COLORS[dataset], alpha=0.12, linewidth=0)
ax_answered.set(title="Excluded-class images answered (lower is better)", ylabel="share answered", ylim=(0, None))
percent(ax_answered)
ax_auroc.axhline(0.5, color=INK_MUTED, linestyle=":", linewidth=1.5)
ax_auroc.set(title="Separating in-set from excluded-class images", ylabel="AUROC", ylim=(0.4, 1.0))
for ax in (ax_answered, ax_auroc):
    k_axis(ax)
    ax.set_xticks(range(2, 10))
legend(ax_answered, styles=(("-", "softmax 95% threshold"), ("--", "SDM α ≥ 0.95")), loc="upper right")
legend(ax_auroc, loc="lower right")
plt.tight_layout()
plt.show()

runs_ood, runs = runs, ood  # table() summarizes `runs`; point it at the K < 10 runs for this table
ood_table = table({"ood_softmax_answered_plain_95": "softmax 95% answers", "ood_sdm_answered_095": "SDM α≥0.95 answers",
                   "ood_sdm_answered_any": "SDM any-region answers", "ood_sdm_flag_rate": "SDM flags OOD",
                   "ood_softmax_auroc": "softmax AUROC", "ood_sdm_auroc": "SDM AUROC"},
                  {"ood_softmax_auroc": "{:.3f}", "ood_sdm_auroc": "{:.3f}"})
runs = runs_ood
ood_table

## Interpretation after corrected evaluation

Earlier saved results used order-dependent tied-score metrics and selected the matched-coverage threshold using SDM validation coverage. Those historical results cannot support conclusions about the corrected evaluator. They remain in Git history; this notebook requires the current `evaluation_version` and its outputs are cleared.

After corrected runs finish, report coverage alongside selective accuracy, distinguish calibration-matched rules from equal validation coverage, and report absent SDM regions explicitly. Compare chains and class identities before attributing changes solely to the number of classes. Bootstrap intervals condition on the trained models and fitted rules; they do not measure training-seed uncertainty or establish significance across the full set of explored comparisons.
